# Exercise on Joins and anti-joins: add information from other tables

In [1]:
# import libraries - solution
import pandas as pd
import numpy as np

# Set some Pandas options: maximum number of rows/columns it's going to display
#pd.set_option('display.max_rows', 1000)
#pd.set_option('display.max_columns', 100)

## Load data from clinical trial

Data comes in two different files. The file `predimed_records.csv` file contains the clinical data for each patient, except which diet group they were assigned. The file `predimed_mapping.csv` contain the information of which patient was assigned to which diet group. 

In [28]:
# solution
df = pd.read_csv('../../data/predimed_records.csv')
df.head()

,patient-id,location-id,sex,age,smoke,bmi,waist,wth,htn,diab,hyperchol,famhist,hormo,p14,toevent,event
0,436,4,Male,58,Former,33.53,122,0.753086,No,No,Yes,No,No,10,5.374401,Yes
1,1130,4,Male,77,Current,31.05,119,0.730061,Yes,Yes,No,No,No,10,6.097194,No
2,1131,4,Female,72,Former,30.86,106,0.654321,No,Yes,No,Yes,No,8,5.946612,No
3,1132,4,Male,71,Former,27.68,118,0.694118,Yes,No,Yes,No,No,8,2.907598,Yes
4,1111,2,Female,79,Never,35.94,129,0.806250,Yes,No,Yes,No,No,9,4.761123,No


In [3]:
# solution
df_info = pd.read_csv('../../data/predimed_mapping.csv')
df_info.head()

,location-id,patient-id,group
0,2,885,MedDiet + VOO
1,1,182,MedDiet + Nuts
2,1,971,MedDiet + Nuts
3,2,691,MedDiet + Nuts
4,2,632,Control


There were 5 different locations where the study was conducted, each one gave an identification number `patient-id` to each participant.

In [4]:
# solution
df_info['location-id'].unique()

array([2, 1, 3, 4, 5])

## 1. Add diet information to the patients' records

* For how many patients do we have clinical information? (i.e., rows in `df`)
* For how many patients do we have diet information? (i.e., rows in `info`)

In [5]:
# solution
len(df)

6324

In [6]:
# solution
len(df_info)

6287

### **Exercise** : 

Combine the two tables into one, where all patient informaiton is available.

In [7]:
# solution

# Explore the date
len(df['patient-id'].unique())

1324

If there are only 1324 unique patient-ids, we can't merge only based on the patient-id because there are multiple rows for the same patient. We need to use the location-id as well.

In [8]:
# solution

len(df['location-id'].unique())

5

In [9]:
# solution

1324*5

6620

Looks like not all patients have been tested at all locations.

#### Solution - naive O(n*m)

In [10]:
# solution
# final check - check if the indices are consequitive


In [29]:
# solution

# add a column group
df.insert(0, 'group_2_for_loops', '')

In [30]:
# %%timeit
# solution

# 2 for loops

groups = []

for patient in df['patient-id'].unique():
    df_patient = df[df['patient-id'] == patient]
    for loc in df_patient['location-id']:
        row_df_info = df_info[(df_info['patient-id'] == patient) &
                            (df_info['location-id'] == loc)]
        if len(row_df_info) == 0:
            group = np.nan
        if len(row_df_info) == 1:
            group = row_df_info.group.values[0]
        indx_df = df[(df['patient-id'] == patient) &
                            (df['location-id'] == loc)].index.tolist()
        df.loc[indx_df, 'group_2_for_loops'] = group


#### Solution - a bit better

In [31]:
# %%timeit
# solution

# For loop over patient-id

groups = []
count = 0
for i, patient in enumerate(df['patient-id']):
    loc = df['location-id'][i]
    row_df_info = df_info[(df_info['patient-id'] == patient) &
                            (df_info['location-id'] == loc)]
    if len(row_df_info) == 0:
        group = np.nan
        count+=1
    if len(row_df_info) == 1:
        group = row_df_info.group.values[0]
    if len(row_df_info) > 1:
        print(f'error repeated rows for patient {patient} at {loc}')

    groups.append(group)

In [23]:
# solution

# check if the numebr of nans is correct
len(df_info) + count

6324

In [25]:
# solution

# check if the groups list is as long as the dataframe

print(len(groups))
print(len(df))

6324
6324


In [32]:
# solution 

# insert the group column
df.insert(len(df.columns), 'group', groups)

In [37]:
df[df['group'] != df['group_2_for_loops']]

,group_2_for_loops,patient-id,location-id,sex,age,smoke,bmi,waist,wth,htn,diab,hyperchol,famhist,hormo,p14,toevent,event,group
69,NaN,402,3,Male,72,Former,31.24,103,0.651899,Yes,No,Yes,No,NaN,7,3.616701,No,NaN
274,NaN,292,3,Male,70,Former,21.35,80,0.516129,No,Yes,No,No,No,9,3.759069,Yes,NaN
406,NaN,583,3,Male,67,Current,30.48,97,0.602484,Yes,No,Yes,No,No,10,6.559890,No,NaN
511,NaN,883,1,Female,55,Never,34.24,103,0.715278,Yes,Yes,Yes,No,No,10,6.132786,No,NaN
559,NaN,915,2,Female,64,Never,33.06,114,0.690909,Yes,No,Yes,No,No,11,5.708419,No,NaN
609,NaN,830,2,Male,62,Current,25.78,107,0.668750,Yes,No,Yes,No,No,11,3.928816,No,NaN
766,NaN,675,4,Female,67,Never,29.79,93,0.636986,Yes,No,Yes,Yes,No,12,5.347023,No,NaN
796,NaN,382,5,Female,73,Never,28.69,98,0.680556,No,No,Yes,No,No,8,4.394250,No,NaN
887,NaN,312,5,Female,67,Never,30.77,96,0.653061,Yes,No,Yes,No,No,10,1.787817,No,NaN
1025,NaN,989,3,Female,72,Never,34.48,101,0.687075,Yes,No,Yes,No,No,11,4.887064,No,NaN


#### Solution - O(n+m)

In [41]:
# %%timeit
# solution - optimal
df_with_info = df.merge(df_info, on = ['patient-id', 'location-id'], how = 'left')

In [43]:
# solution
len(df_with_info)

6324

# 4. Save final result in `processed_data_predimed.csv`

1. Using the `.to_csv` method of Pandas DataFrames

In [19]:
df_without_dropped.to_csv('processed_data_predimed.csv', index=None)